## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.base import clone

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", None)

## 2. Import Kaggle

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/kathuman/housing/housing.csv")
ALVO = "median_house_value"

print(f"Registros: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")
print(f"Variável alvo: {ALVO}")

## 3. Visualização Banco


In [ ]:
df.head(20)

In [ ]:
print("=" * 52)
print("DIMENSÕES E TIPOS DE DADOS")
print("=" * 52)
print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")
print()
print("Quantidade de colunas por tipo:")
print(df.dtypes.value_counts())
print()

numericas = df.select_dtypes(include=[np.number]).columns.tolist()
categoricas = df.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Colunas numéricas ({len(numericas)}): {numericas}")
print(f"Colunas categóricas ({len(categoricas)}): {categoricas}")

In [ ]:
df.info()

### Descrição dos atributos

| Atributo | Descrição |
|---|---|
| longitude | Posição geográfica a oeste, em graus |
| latitude | Posição geográfica ao norte, em graus |
| housing_median_age | Idade mediana das casas do bloco |
| total_rooms | Total de cômodos no bloco |
| total_bedrooms | Total de quartos no bloco |
| population | População do bloco |
| households | Quantidade de domicílios do bloco |
| median_income | Renda mediana dos domicílios, em dezenas de milhares de dólares |
| ocean_proximity | Proximidade do oceano, atributo categórico |
| **median_house_value** | **Variável alvo. Valor mediano das casas do bloco, em dólares** |

## 4. Estatísticas descretivas

In [ ]:
df.describe().T

In [ ]:
escalas = df[numericas].describe().T[["min", "max", "mean", "std"]]
escalas["amplitude"] = escalas["max"] - escalas["min"]
escalas["coef_variacao_%"] = (escalas["std"] / escalas["mean"].abs() * 100).round(2)
escalas.sort_values("amplitude", ascending=False)

**Leitura dos resultados:** observa-se disparidade expressiva entre as escalas dos atributos. Variáveis de contagem por bloco, como population, total_rooms, total_bedrooms e households, atingem milhares de unidades, enquanto median_income varia em uma faixa estreita e longitude e latitude assumem valores de poucas dezenas de graus. Essas mesmas variáveis de contagem também apresentam a maior dispersão relativa, com máximos muito distantes da mediana. Como o SVR calcula distâncias entre pontos no espaço de atributos, variáveis de maior magnitude dominariam o modelo, o que torna a padronização obrigatória. Outro ponto a observar é que o valor mediano das casas aparece limitado em um teto de cerca de 500 mil dólares, característica conhecida desta base.

## 5. Valores ausentes e duplicados

In [ ]:
tabela_nulos = pd.DataFrame({
    "valores_ausentes": df.isnull().sum(),
    "percentual": (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values("valores_ausentes", ascending=False)

print(tabela_nulos)
print(f"\nTotal de valores ausentes: {df.isnull().sum().sum()}")
print(f"Linhas completas: {df.dropna().shape[0]} de {df.shape[0]}")
print(f"Linhas duplicadas: {df.duplicated().sum()}")

**Justificativa da escolha:** a coluna total_bedrooms apresentou 207 valores ausentes, cerca de 1% da base. Por ser um percentual pequeno, optou-se pela imputação pela mediana, que mantém todos os registros e é menos sensível a valores atípicos do que a média, cuja presença é esperada em variáveis de contagem como esta. A remoção dos registros também seria viável, mas descartaria dados sem necessidade. A comparação entre remoção de linhas, remoção da coluna, imputação pela média e imputação pela mediana, será feito futuramente.

In [ ]:
def tratar_ausentes(d):
    d = d.copy()
    for c in d.columns:
        if d[c].isnull().any():
            if not pd.api.types.is_numeric_dtype(d[c]):
                d[c] = d[c].fillna(d[c].mode()[0])
            else:
                d[c] = d[c].fillna(d[c].median())
    return d

df_aus = tratar_ausentes(df)
print(f"Valores ausentes restantes: {df_aus.isnull().sum().sum()}")
print(f"Registros preservados: {df_aus.shape[0]}")

## 6. Análise exploratória e Distribuição da variável alvo

In [ ]:
plt.hist(df[ALVO], bins=30, color="#2980b9", edgecolor="white")
plt.axvline(df[ALVO].mean(), color="red", linestyle="--", label="Média")
plt.axvline(df[ALVO].median(), color="green", linestyle="--", label="Mediana")
plt.xlabel("Valor mediano das casas")
plt.ylabel("Quantidade")
plt.title("Distribuição do valor das casas")
plt.legend()
plt.show()

print(f"Assimetria (skewness) do alvo: {df[ALVO].skew():.2f}")
print(f"Valor máximo: {df[ALVO].max():,.0f}")
print(f"Registros no valor máximo: {(df[ALVO] == df[ALVO].max()).sum()}")

### Diagramas de caixa

In [ ]:
n_cols = (len(numericas) + 1) // 2
fig, axes = plt.subplots(2, n_cols, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numericas):
    sns.boxplot(y=df[col], ax=axes[i], color="#16a085")
    axes[i].set_title(col, fontsize=11)
    axes[i].set_ylabel("")

for j in range(len(numericas), len(axes)):
    axes[j].axis("off")

plt.suptitle("Diagramas de caixa das variáveis numéricas", fontsize=14)
plt.tight_layout()
plt.show()

### Correlação

In [ ]:
correlacao = df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
mascara = np.triu(np.ones_like(correlacao, dtype=bool))
sns.heatmap(correlacao, mask=mascara, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, square=True, linewidths=0.5)
plt.title("Matriz de correlação", fontsize=13)
plt.tight_layout()
plt.show()

print("Correlação de cada atributo com o valor das casas:\n")
print(correlacao[ALVO].drop(ALVO).sort_values(ascending=False))

**Leitura dos resultados:** a variável median_income é a mais fortemente correlacionada com o valor das casas (0,69), sendo o principal preditor da base. Todas as demais apresentam correlação fraca, com valor absoluto abaixo de 0,15, entre elas total_rooms (0,13), housing_median_age (0,11) e latitude (negativa, 0,14). Há forte redundância entre total_rooms, total_bedrooms, population e households, que variam juntas porque todas medem o tamanho do bloco. A distribuição do alvo tem cauda à direita, com assimetria de 0,98, e 965 registros acumulados no valor máximo, o que sinaliza o teto da base. Como longitude e latitude se relacionam com o valor de forma não linear, por meio da localização geográfica, é esperado que um kernel não linear tenha vantagem.

## 7. Identificação de outliers

Os valores extremos são identificados pelo método do intervalo interquartil (IQR), que considera atípico todo valor abaixo de Q1 menos 1,5 vezes o IQR ou acima de Q3 mais 1,5 vezes o IQR. Nesta etapa os outliers são apenas identificados, sem nenhuma remoção.

In [ ]:
def relatorio_outliers(d, cols):
    linhas = []
    for c in cols:
        q1, q3 = d[c].quantile(0.25), d[c].quantile(0.75)
        iqr = q3 - q1
        li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n = ((d[c] < li) | (d[c] > ls)).sum()
        linhas.append([c, n, round(100 * n / len(d), 2),
                       d[c].min(), d[c].max(), li, ls])
    return pd.DataFrame(linhas, columns=["variavel", "qtd_outliers", "percentual",
                                         "minimo", "maximo", "lim_inferior", "lim_superior"])

relatorio = relatorio_outliers(df_aus, numericas)
relatorio

## 8. Tratamento dos outliers

Valores extremos nem sempre são erros. Em total_rooms, total_bedrooms, population e households, eles correspondem a blocos grandes e densamente povoados, que são registros reais. Por isso a remoção foi descartada e adotou-se o capping (winsorization) pelos limites do IQR nos atributos preditores, que limita os extremos sem perder registros. Longitude e latitude foram mantidas sem alteração, por serem coordenadas geográficas. A variável alvo também foi mantida, pois alterar o valor das casas distorceria a grandeza que o modelo precisa prever.

**Justificativa da escolha:** o método IQR identificou outliers principalmente nas variáveis de contagem por bloco, conforme a tabela da seção 7. Esses valores não aparentam ser erros de cadastro, e sim blocos de grande porte. O capping reduz a influência deles sobre o treinamento sem descartar nenhum registro, o que atende à regra de não excluir grande parte da base.

In [ ]:
def capping_iqr(d, cols):
    d = d.copy()
    for c in cols:
        q1, q3 = d[c].quantile(0.25), d[c].quantile(0.75)
        iqr = q3 - q1
        d[c] = d[c].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return d
    
preditores_num = [c for c in numericas if c not in [ALVO, "longitude", "latitude"]]
df_out = capping_iqr(df_aus, preditores_num)

print(f"Registros antes: {df_aus.shape[0]}")
print(f"Registros depois: {df_out.shape[0]}")
print("Nenhum registro foi removido, apenas limitado pelos limites do IQR.")

## 9. Preparação dos dados

In [ ]:
def preparar(d):
    X = d.drop(ALVO, axis=1)
    y = d[ALVO]
    X = pd.get_dummies(X, drop_first=True, dtype=float)
    return train_test_split(X, y, test_size=0.20, random_state=42)

def experimento(d, kernel="rbf", C=1, epsilon=0.1, gamma="scale",
                padronizar=True, escalar_alvo=False):
    X_tr, X_te, y_tr, y_te = preparar(d)
    if padronizar:
        sc = StandardScaler()              # ajustado apenas no treino
        X_tr = sc.fit_transform(X_tr)
        X_te = sc.transform(X_te)
    m = SVR(kernel=kernel, C=C, epsilon=epsilon, gamma=gamma)
    if escalar_alvo:
        m = TransformedTargetRegressor(regressor=m, transformer=StandardScaler())
    m.fit(X_tr, y_tr)
    return r2_score(y_te, m.predict(X_te))

X_train, X_test, y_train, y_test = preparar(df_out)
print(f"Treino: {X_train.shape[0]} registros")
print(f"Teste: {X_test.shape[0]} registros")
print(f"Atributos após codificação: {X_train.shape[1]}")

## 10. Modelo base e comparação antes e depois do tratamento

In [ ]:
resultados = [
    ["Baseline", "Nenhum", "rbf", 1, 0.1, experimento(df.dropna(), padronizar=False)],
    ["Modelo 2", "Valores ausentes", "rbf", 1, 0.1, experimento(df_aus, padronizar=False)],
    ["Modelo 3", "Outliers", "rbf", 1, 0.1, experimento(df_out, padronizar=False)],
    ["Modelo 4", "Outliers + padronização de X", "rbf", 1, 0.1, experimento(df_out, padronizar=True)],
    ["Modelo 5", "Outliers + padronização de X e y", "rbf", 1, 0.1,
     experimento(df_out, padronizar=True, escalar_alvo=True)],
]

tabela = pd.DataFrame(resultados, columns=["Modelo", "Tratamento", "Kernel", "C", "Epsilon", "R²"])
tabela["R²"] = tabela["R²"].round(4)
tabela

**Explicação metodológica:** o SVR é sensível à escala não apenas dos atributos, mas também da variável alvo. Com o valor das casas na casa das centenas de milhares de dólares, os parâmetros C e epsilon ficam em escala inadequada em relação ao alvo, e o modelo tende a prever valores quase constantes, com R² próximo de zero. A padronização dos atributos e do alvo corrige essa distorção, e é por isso que a tabela acima mostra o maior salto de R² nos Modelos 4 e 5.

### Comparação de estratégias para os valores ausentes

A coluna total_bedrooms possui valores ausentes, e quatro estratégias são comparadas: remoção das linhas incompletas, remoção da coluna, imputação pela média e imputação pela mediana. Em cada caso o capping de outliers é aplicado depois do tratamento, e o modelo é o mesmo SVR com kernel rbf, C igual a 1 e epsilon igual a 0,1, com atributos e alvo padronizados. Assim, a única diferença entre os experimentos é a estratégia aplicada aos valores ausentes.

In [ ]:
def tratar_estrategia(d, metodo):
    d = d.copy()
    cols_cap = [c for c in preditores_num]

    if metodo == "Remover linhas":
        d = d.dropna()
    elif metodo == "Remover coluna":
        d = d.drop(columns=["total_bedrooms"])
        cols_cap = [c for c in cols_cap if c != "total_bedrooms"]
    elif metodo == "Imputar média":
        d["total_bedrooms"] = d["total_bedrooms"].fillna(d["total_bedrooms"].mean())
    elif metodo == "Imputar mediana":
        d["total_bedrooms"] = d["total_bedrooms"].fillna(d["total_bedrooms"].median())

    return capping_iqr(d, cols_cap)

print("=" * 52)
print("VALORES USADOS NA IMPUTAÇÃO DE total_bedrooms")
print("=" * 52)
print(f"Média:   {df['total_bedrooms'].mean():.2f}")
print(f"Mediana: {df['total_bedrooms'].median():.2f}")
print()

linhas = []
for metodo in ["Remover linhas", "Remover coluna", "Imputar média", "Imputar mediana"]:
    d_m = tratar_estrategia(df, metodo)
    r2_m = experimento(d_m, kernel="rbf", C=1, epsilon=0.1, padronizar=True, escalar_alvo=True)
    linhas.append([metodo, d_m.shape[0], d_m.shape[1], round(r2_m, 4)])

comparacao_ausentes = pd.DataFrame(linhas, columns=["Estratégia", "Registros", "Colunas", "R²"])
comparacao_ausentes

**Leitura dos resultados:** as quatro estratégias produzem valores de R² muito próximos, o que era esperado, já que apenas cerca de 1% da base é afetado. A remoção da coluna tem pouco impacto porque total_bedrooms é redundante com total_rooms, households e population. A imputação pela média e pela mediana diferem pouco porque os outliers foram limitados pelo capping, mas a mediana permanece a escolha mais segura por ser menos sensível a valores extremos. Adotou-se a imputação pela mediana por preservar todos os registros e todas as colunas, sem perda de informação.

### Efeito isolado do tratamento de outliers

A comparação dos Modelos 2 e 3 foi feita quando o R² ainda era negativo, o que impede qualquer conclusão. Por isso o capping é comparado novamente, agora com os atributos e o alvo padronizados, mantendo todo o resto igual.

In [ ]:
r2_sem_cap = experimento(df_aus, padronizar=True, escalar_alvo=True)
r2_com_cap = experimento(df_out, padronizar=True, escalar_alvo=True)

print("=" * 52)
print("EFEITO DO CAPPING COM PADRONIZAÇÃO DE X E y")
print("=" * 52)
print(f"R² sem capping: {r2_sem_cap:.4f}")
print(f"R² com capping: {r2_com_cap:.4f}")
print(f"Diferença: {r2_com_cap - r2_sem_cap:+.4f}")

**Leitura dos resultados:** com os atributos e o alvo padronizados, o R² foi de 0,7474 sem capping e de 0,7487 com capping, uma diferença de apenas 0,0013. O efeito é marginal e, por ter sido medido em uma única divisão entre treino e teste, não é suficiente para afirmar que o tratamento melhorou o modelo. Uma possível explicação é que o kernel rbf, combinado à padronização, já é pouco afetado por valores extremos nos atributos. O capping foi mantido por reduzir a influência de blocos de porte muito acima do usual sem descartar nenhum registro, mas o ganho de desempenho atribuível a ele é desprezível

## 11. Comparação de kernels

Os kernels linear, rbf e poly são comparados com os demais parâmetros mantidos fixos.

In [ ]:
print("=" * 52)
print("COMPARAÇÃO DE KERNELS")
print("=" * 52)

res_kernel = []
for k in ["linear", "rbf", "poly"]:
    r2_k = experimento(df_out, kernel=k, C=1, epsilon=0.1, escalar_alvo=True)
    res_kernel.append([k, round(r2_k, 4)])
    print(f"  {k}: R² = {r2_k:.4f}")

pd.DataFrame(res_kernel, columns=["Kernel", "R²"])

**Leitura dos resultados:** o kernel rbf obteve o melhor R² (0,7487), seguido do poly (0,6814) e do linear (0,6147). A vantagem dos kernels não lineares é coerente com a análise exploratória, pois o valor das casas depende da localização geográfica de forma não linear, o que um modelo linear não consegue representar.

## 12. GridSearch


In [ ]:
pipe = TransformedTargetRegressor(
    regressor=Pipeline([("scaler", StandardScaler()), ("svr", SVR())]),
    transformer=StandardScaler()
)

parametros = {
    "regressor__svr__C": [0.1, 1, 10, 100],
    "regressor__svr__epsilon": [0.01, 0.1, 0.5],
    "regressor__svr__gamma": ["scale", "auto"],
    "regressor__svr__kernel": ["rbf", "linear"],
}

# amostra de treino apenas para a busca, por causa do custo do SVR
X_amostra = X_train.sample(n=min(4000, len(X_train)), random_state=42)
y_amostra = y_train.loc[X_amostra.index]

grid = GridSearchCV(pipe, parametros, cv=3, scoring="r2", n_jobs=-1)
grid.fit(X_amostra, y_amostra)

print("=" * 52)
print("RESULTADO DA BUSCA EM GRADE")
print("=" * 52)
print("Melhores parâmetros:")
for p, v in grid.best_params_.items():
    print(f"  {p.split('__')[-1]}: {v}")
print(f"\nR² médio na validação cruzada: {grid.best_score_:.4f}")

# retreino do melhor modelo com todo o conjunto de treino
modelo_final = clone(grid.best_estimator_)
modelo_final.fit(X_train, y_train)
print("\nModelo final retreinado com todo o conjunto de treino.")

**Leitura dos resultados:** a busca em grade selecionou o kernel rbf com C igual a 10, epsilon igual a 0,1 e gamma igual a scale, com R² médio de 0,7498 na validação cruzada. O aumento de C de 1 para 10 permite ao modelo ajustar-se mais aos dados de treino, o que melhorou o desempenho em relação ao SVR com C igual a 1.

## 13. Avaliação final

O modelo final é avaliado no conjunto de teste. São calculados R², MAE e RMSE. O R² do conjunto de teste é o valor que atesta o cumprimento do requisito mínimo.

In [ ]:
y_pred = modelo_final.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 52)
print("DESEMPENHO NO CONJUNTO DE TESTE")
print("=" * 52)
print(f"R²:   {r2:.4f}")
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"\nAtingiu R² > 0,65? {'Sim' if r2 > 0.65 else 'Não'}")

### Valores reais e valores previstos

Gráfico de dispersão entre o valor real e o valor previsto pelo modelo.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.3, color="#2980b9", s=10)
lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lim, lim, "r--", label="Predição perfeita")
plt.xlabel("Valor real")
plt.ylabel("Valor previsto")
plt.title("Valores reais × valores previstos")
plt.legend()
plt.show()

**Leitura dos resultados:** os pontos acompanham a diagonal para valores reais até cerca de 300 mil dólares, com dispersão crescente nas faixas mais altas, o que indica maior erro em imóveis de maior valor. Observa-se uma coluna vertical de pontos em 500 mil dólares, que corresponde ao teto da base: nesses registros o valor real está limitado, e as previsões variam de cerca de 150 mil a 570 mil, com predominância de subestimação. Há também alguns pontos isolados longe da diagonal e uma previsão levemente negativa, que não faz sentido para um preço. O MAE de 36.422,83 equivale a cerca de 17,6% do valor médio das casas, e o RMSE de 55.155,71 é maior que o MAE, o que indica que existem erros grandes pesando mais que a média.

## 14. Tabela de experimentos

Consolidação de todos os experimentos realizados, incluindo o modelo final obtido pela busca em grade.

In [ ]:
bp = grid.best_params_
final = ["Modelo final", "Dados tratados + padronização + GridSearch",
         bp["regressor__svr__kernel"], bp["regressor__svr__C"],
         bp["regressor__svr__epsilon"], round(r2, 4)]

tabela_final = pd.concat([tabela, pd.DataFrame([final], columns=tabela.columns)],
                         ignore_index=True)
tabela_final

## 15. Conclusão

1. **O modelo atingiu R² > 0,65?** Sim. O modelo final obteve R² de 0,7678 no conjunto de teste, com MAE de 36.422,83 e RMSE de 55.155,71.

2. **Qual tratamento teve maior impacto?** A padronização da variável alvo. Os modelos 1 a 4 tiveram R² negativo, entre -0,0546 e -0,0425, e o R² passou para 0,7487 quando o alvo também foi padronizado. Esse resultado é coerente com o enunciado, que indica que o SVR é sensível à escala.

3. **O tratamento dos outliers melhorou o resultado?** De forma marginal. Com atributos e alvo padronizados, o R² foi de 0,7474 sem capping e de 0,7487 com capping, uma diferença de 0,0013. Nenhum registro foi removido. Nos valores ausentes, as quatro estratégias testadas (remover linhas, remover coluna, imputar pela média e imputar pela mediana) ficaram entre 0,7459 e 0,7550, e foi adotada a mediana.

4. **Qual kernel apresentou melhor resultado?** O rbf, com R² de 0,7487, contra 0,6814 do poly e 0,6147 do linear.

5. **Quais hiperparâmetros foram utilizados no modelo final?** Kernel rbf, C igual a 10, epsilon igual a 0,1 e gamma igual a scale, com atributos e alvo padronizados, escolhidos pelo GridSearchCV (R² médio de 0,7498 na validação cruzada).

6. **Quais dificuldades foram encontradas?** O R² negativo nos modelos iniciais; o custo computacional do SVR em mais de 20 mil registros, que levou a executar o GridSearch em uma amostra de 4.000 registros do treino e a retreinar o melhor modelo com todo o treino; e o teto de 500.001 na variável alvo, que se repete em 965 registros. Como limitações, a imputação e o capping foram calculados com a base completa antes da divisão treino e teste, e as comparações de tratamentos e kernels foram medidas no conjunto de teste.
